# Experiment 1 — Baseline Logistic Regression
Telco Customer Churn — MLOps Project

Baseline model with simple label-encoding preprocessing, tracked with MLflow (via DagsHub).

In [1]:
import pandas as pd
import numpy as np
import time
import logging

import mlflow
import mlflow.sklearn
import dagshub

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

## 1. Load data
Replace `DATA_URL` with the raw GitHub URL of your uploaded CSV.

In [2]:
# TODO: replace with your actual raw GitHub URL once the dataset is uploaded
# DATA_URL = "https://raw.githubusercontent.com/<your-username>/<your-repo>/main/data/WA_Fn-UseC_-Telco-Customer-Churn.csv"

# df = pd.read_csv(DATA_URL)
df = pd.read_csv('data.csv')
df.to_csv('data.csv', index=False)
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [4]:
df.isnull().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

## 2. Preprocessing

For this baseline experiment we keep it simple: drop the ID column, fix `TotalCharges`, encode the target, and label-encode the remaining categoricals. More careful encoding (one-hot, scaling, etc.) can come in later experiments.

In [5]:
def preprocess_data(df):
    """Clean and encode the Telco churn dataframe for the baseline model."""
    df = df.copy()

    # Drop identifier column (not predictive)
    df.drop(columns=['customerID'], inplace=True, errors='ignore')

    # TotalCharges is stored as object with some blank strings -> coerce to numeric
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
    df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)

    # Encode target: Yes/No -> 1/0
    df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

    # Label-encode remaining categorical columns
    cat_cols = df.select_dtypes(include='object').columns
    for col in cat_cols:
        df[col] = LabelEncoder().fit_transform(df[col])

    return df

df = preprocess_data(df)
df.head()

C:\Users\Hp\AppData\Local\Temp\ipykernel_7348\2902682083.py:10: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,0,1,0,1,0,1,0,0,2,0,0,0,0,0,1,2,29.85,29.85,0
1,1,0,0,0,34,1,0,0,2,0,2,0,0,0,1,0,3,56.95,1889.50,0
2,1,0,0,0,2,1,0,0,2,2,0,0,0,0,0,1,3,53.85,108.15,1
3,1,0,0,0,45,0,1,0,2,0,2,2,0,0,1,0,0,42.30,1840.75,0
4,0,0,0,0,2,1,0,1,0,0,0,0,0,0,0,1,2,70.70,151.65,1


In [6]:
# sanity check: no missing values, target is binary
df.isnull().sum()

gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

## 3. Train / test split

In [7]:
X = df.drop(columns=['Churn'])
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train.shape, X_test.shape

((5634, 19), (1409, 19))

## 4. MLflow / DagsHub setup

Replace `repo_owner` / `repo_name` with your own DagsHub repo for this project.

In [8]:
mlflow.set_tracking_uri('https://dagshub.com/rehansarfraz8903/Customer-Churn-prediction.mlflow')
dagshub.init(repo_owner='rehansarfraz8903', repo_name='Customer-Churn-prediction', mlflow=True)

mlflow.set_experiment("Logistic Regression Baseline")

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

c:\Users\Hp\miniconda3\envs\churn\lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" for Jupyter
support
  warnings.warn('install "ipywidgets" for Jupyter support')



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=d0bae87e-19a7-492a-a95a-00dacba56850&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=c6504e507d5cd75acac4092cb4f56f2ce1c8158ea2ee83d3c8ac99f65f35d814




2026-08-12 23:22:28,281 - INFO - HTTP Request: POST https://dagshub.com/login/oauth/middleman "HTTP/1.1 200 OK"


2026-08-12 23:22:29,873 - INFO - HTTP Request: POST https://dagshub.com/login/oauth/access_token "HTTP/1.1 200 OK"
2026-08-12 23:22:31,201 - INFO - HTTP Request: GET https://dagshub.com/api/v1/user "HTTP/1.1 200 OK"


...

Initialized MLflow to track repo "rehansarfraz8903/Customer-Churn-prediction"

2026-08-12 23:22:33,357 - INFO - Initialized MLflow to track repo "rehansarfraz8903/Customer-Churn-prediction"


Repository rehansarfraz8903/Customer-Churn-prediction initialized!

2026-08-12 23:22:33,377 - INFO - Repository rehansarfraz8903/Customer-Churn-prediction initialized!
2026/08/12 23:22:34 INFO mlflow.tracking.fluent: Experiment with name 'Logistic Regression Baseline' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/fb6e34d06d8741fcb6e0bb011c9325f8', creation_time=1786602155495, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1786602155495, lifecycle_stage='active', name='Logistic Regression Baseline', tags={}, trace_location=None, workspace='default'>

## 5. Train baseline model + log to MLflow

In [9]:
logging.info("Starting MLflow run...")

with mlflow.start_run():
    start_time = time.time()

    try:
        logging.info("Logging preprocessing parameters...")
        mlflow.log_param("preprocessing", "label_encoding")
        mlflow.log_param("test_size", 0.2)
        mlflow.log_param("num_features", X_train.shape[1])

        logging.info("Initializing Logistic Regression model...")
        model = LogisticRegression(max_iter=1000)

        logging.info("Fitting the model...")
        model.fit(X_train, y_train)
        logging.info("Model training complete.")

        mlflow.log_param("model", "Logistic Regression")

        logging.info("Making predictions...")
        y_pred = model.predict(X_test)

        logging.info("Calculating evaluation metrics...")
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        logging.info("Logging evaluation metrics...")
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        logging.info("Saving and logging the model...")
        mlflow.sklearn.log_model(model, "model")

        end_time = time.time()
        logging.info(f"Model training and logging completed in {end_time - start_time:.2f} seconds.")

        logging.info(f"Accuracy: {accuracy}")
        logging.info(f"Precision: {precision}")
        logging.info(f"Recall: {recall}")
        logging.info(f"F1 Score: {f1}")

    except Exception as e:
        logging.error(f"An error occurred: {e}", exc_info=True)

2026-08-12 23:22:49,784 - INFO - Starting MLflow run...
2026-08-12 23:22:50,963 - INFO - Logging preprocessing parameters...
2026-08-12 23:22:52,191 - INFO - Initializing Logistic Regression model...
2026-08-12 23:22:52,193 - INFO - Fitting the model...
c:\Users\Hp\miniconda3\envs\churn\lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
2026-08-12 23:22:54,575 - INFO - Model training complete.
2026-08-12 23:22:55,059 - INFO - Making predictions...
2026-08-12 23:22:55,070 - INFO - C

🏃 View run rebellious-bear-51 at: https://dagshub.com/rehansarfraz8903/Customer-Churn-prediction.mlflow/#/experiments/0/runs/43775f84d2a046f78a4269145302f14a
🧪 View experiment at: https://dagshub.com/rehansarfraz8903/Customer-Churn-prediction.mlflow/#/experiments/0


## 6. (Optional) Confusion matrix — quick sanity check

In [10]:
confusion_matrix(y_test, y_pred)

array([[919, 116],
       [167, 207]])